# G1, Replay-IS + DAgger: nên bắt chước thầy MPC trong bao lâu? (Kaggle, GPU)

Hiện tại "thầy" MPC gắn nhãn mỗi 5 vòng **trong 110 vòng đầu**, trọng số bắt chước (BC) giảm từ 1
về 0 tại **vòng 150**, nhãn cũ hơn 50 vòng bị bỏ. Thí nghiệm này hỏi: **kéo dài BC có giúp không?**

| Biến thể | Gắn nhãn tới vòng | BC giảm về | Nhãn giữ | Tham số thêm |
|---|---|---|---|---|
| **Base** (gốc) | 110 | 0 tại vòng 150 | 50 vòng | (không) |
| **BC-300** | 300 | 0 tại vòng 340 | 50 vòng | `collect-iterations 300`, `bc-iterations 340` |
| **BC-floor** | 110 | 0,1 mãi mãi | mãi mãi (10 lần gắn nhãn cuối) | `bc-floor 0.1`, `max-age None` |

Mọi biến thể dùng task `Mpopi-G1-2k-Replay-IS-DAgger` (4096 env, 2000 vòng, lệnh tới 1,5 m/s), **2 seed**,
và **đo Δ** (thầy hơn trò bao nhiêu) ở mọi lần gắn nhãn với phép đo đã sửa lỗi quan sát cũ.
Biến thể **Base** đồng thời kiểm tra code hiện tại cho lại kết quả các lần chạy trước.

**Cách đọc kết quả** (cần thấy ở **cả 2 seed**):

- BC-300 hoặc BC-floor đạt mốc học sớm hơn / sai số thấp hơn Base → thầy còn hữu ích, nên kéo dài BC.
- Không khác biệt → 150 vòng là đủ; phần lợi của BC nằm ở giai đoạn đầu.
- Kém hơn Base → bắt chước lâu cản PPO tự tối ưu reward.

## Cài đặt Kaggle trước khi chạy (panel bên phải → *Settings*)

1. **Accelerator: GPU T4 x2.** Không chọn P100: bản PyTorch của dự án (CUDA 12.8) không hỗ trợ P100.
   Notebook dùng **cả 2 GPU**, mỗi GPU train một lần cùng lúc.
2. **Internet: On** (cần xác minh số điện thoại cho tài khoản Kaggle) để tải code và thư viện.

## Cách chạy

- Import notebook này thành **một notebook mới** trên Kaggle (không lưu đè notebook cũ), rồi
  **Save Version → Save & Run All (Commit)**. 6 lần train ≈ 6,5–7 giờ trên 2 GPU T4.
- Kết quả: `mpc_g1_bc_results.zip` ở tab **Output**.
- Không kịp xong thì notebook bỏ qua phần còn thiếu; thêm output của phiên bản đã chạy làm *Input*
  (*Add Input → Your Work*) rồi chạy lại để train tiếp.

## 0. Cấu hình

In [ ]:
import os
import time

NOTEBOOK_START = time.time()

REPO_URL = "https://github.com/TamasTran/mjlab_MPOPI.git"
BRANCH = "mpc-stage1"
TASK = "Mpopi-G1-2k-Replay-IS-DAgger"
EVAL_TASK = "Mpopi-G1-2k-PPO"  # Cùng môi trường và mạng; không dựng thầy MPC khi đánh giá.
NUM_ENVS = 4096
ITERATIONS = 2000
SEEDS = [1, 2]
# Tham số thêm cho mọi lần train (xem GUIDE.md mục 7), ví dụ đổi vận tốc mục tiêu thành 2,0 m/s:
# EXTRA_ARGS = "--env.curriculum.command-vel.params.velocity-stages.1.lin-vel-x -1.0,2.0"
EXTRA_ARGS = ""

# Biến thể cần train (True/False).
TRAIN = {
  "BC-300": True,
  "Base": True,
  "BC-floor": True,
}
# Gắn nhãn mỗi MPC_EVERY vòng (preset); số vòng gắn nhãn của từng biến thể (để ước tính thời gian).
MPC_EVERY = 5
LABEL_UNTIL = {"BC-300": 300, "Base": 110, "BC-floor": 110}
# Đo thầy có hơn trò không (Δ): cứ mỗi TEACHER_GAP_EVERY bước gắn nhãn, so return của kế
# hoạch MPC với return của policy từ cùng trạng thái. None để tắt (thêm ~12% chi phí MPC).
TEACHER_GAP_EVERY = 4

# Canh giờ (Kaggle dừng phiên GPU sau khoảng 12 giờ).
TIME_BUDGET_HOURS = 11.5  # Tính từ lúc chạy ô này.
EVAL_RESERVE_MIN = 60  # Phút chừa lại cho đánh giá, video và đóng gói.
SMOKE_TEST = True  # Chạy thử 3 vòng mỗi phương pháp và ước tính thời gian.
EVAL_ENVS = 64  # Đánh giá: 64 robot, 1 giây tăng tốc rồi đo trong 10 giây.
EVAL_SETTLE, EVAL_STEPS = 50, 550
EVAL_SPEEDS = "0.5 1.0 1.5"  # Lệnh vận tốc khi đánh giá (m/s), cách nhau bằng dấu cách.
RECORD_VIDEO = True

REPO_DIR = "/tmp/mjlab_MPOPI"  # Không để trong /kaggle/working (thư mục output).
OUT_ROOT = "/kaggle/working/mpc_g1_bc"
LOG_ROOT = f"{OUT_ROOT}/train_logs"
CONSOLE_DIR = f"{OUT_ROOT}/train_console"
EVAL_DIR = f"{OUT_ROOT}/eval"
for d in (CONSOLE_DIR, EVAL_DIR):
  os.makedirs(d, exist_ok=True)
DEADLINE = NOTEBOOK_START + TIME_BUDGET_HOURS * 3600
TRAIN_DEADLINE = DEADLINE - EVAL_RESERVE_MIN * 60


def remaining_min(deadline=DEADLINE):
  return (deadline - time.time()) / 60


print(f"Thời gian cho train: {remaining_min(TRAIN_DEADLINE):.0f} phút; tổng: {remaining_min():.0f} phút")

In [ ]:
#@title Hàm tiện ích
import subprocess


def run(cmd, cwd=None, quiet_tables=False, check=True):
  """Run a shell command and stream its output."""
  print(f"$ {cmd}", flush=True)
  proc = subprocess.Popen(
    cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env={**os.environ},
  )
  for line in proc.stdout:
    if quiet_tables and ("[INFO]" in line or line.startswith(("|", "+"))):
      continue
    print(line, end="", flush=True)
  if proc.wait() != 0 and check:
    raise RuntimeError(f"Command failed with exit code {proc.returncode}: {cmd}")
  return proc.returncode

## 1. Kiểm tra GPU

In [ ]:
gpu_list = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
print(gpu_list)
GPUS = [line.split(":")[0].split()[-1] for line in gpu_list.splitlines() if line.startswith("GPU")]
if not GPUS:
  raise RuntimeError("Không thấy GPU. Settings → Accelerator → GPU T4 x2.")
if "P100" in gpu_list:
  raise RuntimeError("P100 không được hỗ trợ. Settings → Accelerator → GPU T4 x2.")
print(f"Dùng {len(GPUS)} GPU: {GPUS}")

## 2. Chép kết quả phiên trước (nếu có) và lấy code

In [ ]:
import glob
import shutil

previous = sorted(glob.glob("/kaggle/input/**/mpc_g1_bc", recursive=True))
for prev in previous:
  print("Chép kết quả phiên trước từ", prev)
  shutil.copytree(prev, OUT_ROOT, dirs_exist_ok=True)
done_before = sorted(os.path.basename(p)[:-5] for p in glob.glob(f"{CONSOLE_DIR}/*.done"))
print("Các lần train đã xong:", done_before or "chưa có")

if not REPO_DIR.startswith("/tmp/"):
  raise RuntimeError("REPO_DIR phải nằm trong /tmp (ô này xóa rồi clone lại thư mục đó).")
if os.path.exists(REPO_DIR):
  shutil.rmtree(REPO_DIR)
run(f"git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}")
run("git log --oneline -1", cwd=REPO_DIR)
if not os.path.exists(f"{REPO_DIR}/src/mpopi_train/presets.py"):
  raise RuntimeError("Code trên GitHub đã cũ: hãy push nhánh mpc-stage1 bản mới nhất.")
print("OK")

## 3. Cài đặt môi trường

Lần đầu mất khoảng 5–10 phút.

In [ ]:
run("curl -LsSf https://astral.sh/uv/install.sh | sh")
os.environ["PATH"] = f"{os.path.expanduser('~')}/.local/bin:{os.environ['PATH']}"
run("uv sync --extra cu128", cwd=REPO_DIR)
run(
  "uv run --extra cu128 python -c \"import torch; print('torch', torch.__version__, "
  "'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0))\"",
  cwd=REPO_DIR,
)
UV = "uv run --extra cu128"

## 4. Lệnh train của từng biến thể

In [ ]:
A = "--agent.algorithm.mpopi"
USES_MPC = set(TRAIN)  # Mọi biến thể đều có thầy MPC.
GAP = f"{A}.mpc.teacher-gap-every {TEACHER_GAP_EVERY}" if TEACHER_GAP_EVERY else ""
# Tham số của mỗi biến thể, thêm vào preset Replay-IS + DAgger.
ALL_METHODS = {
  "BC-300": f"{GAP} {A}.mpc.collect-iterations 300 {A}.mpc.bc-iterations 340",
  "Base": GAP,
  "BC-floor": f"{GAP} {A}.mpc.bc-floor 0.1 {A}.mpc.max-age None",
}
METHODS = {m: a for m, a in ALL_METHODS.items() if TRAIN[m]}


def train_cmd(method, seed, iterations=ITERATIONS, log_root=LOG_ROOT, extra=""):
  return (
    f"{UV} mpopi-train {TASK} --agent.logger tensorboard "
    f"--env.scene.num-envs {NUM_ENVS} --agent.max-iterations {iterations} "
    f"--agent.seed {seed} --agent.run-name {method}_s{seed} --log-root {log_root} "
    f"{METHODS[method]} {EXTRA_ARGS} {extra}"
  )


for m in METHODS:
  print(f"# {m}\n{train_cmd(m, SEEDS[0])}\n")

## 5. Chạy thử và ước tính thời gian

Mỗi biến thể chạy 3 vòng trên 4096 env (thu MPC ở cả 3 vòng) để kiểm tra lỗi và đo thời gian
một vòng PPO và một lần thu MPC. Từ đó ước tính thời gian của cả lần train.

In [ ]:
import re

ANSI = re.compile(r"\x1b\[[0-9;]*m")
ESTIMATE_S = {}  # Ước tính thời gian (giây) một lần train đầy đủ của mỗi phương pháp.
if SMOKE_TEST:
  for m in METHODS:
    log = f"{OUT_ROOT}/smoke_{m}.log"
    cmd = train_cmd(m, 0, iterations=3, log_root="/tmp/smoke_logs")
    if m in USES_MPC:  # Thu MPC ở cả 3 vòng chạy thử.
      cmd += f" {A}.mpc.collect-every 1"
    t0 = time.time()
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": GPUS[0]}
    rc = subprocess.run(f"{cmd} > '{log}' 2>&1", shell=True, cwd=REPO_DIR, env=env).returncode
    text = ANSI.sub("", open(log, errors="ignore").read())
    if rc != 0:
      print(text[-3000:])
      raise RuntimeError(f"Chạy thử {m} bị lỗi")
    it_times = [float(x) for x in re.findall(r"Iteration time:\s*([\d.]+)s", text)]
    mpc_s = [float(x) for x in re.findall(r"mpc/collect_seconds loss:\s*([\d.]+)", text)]
    per_it = min(it_times) if it_times else 0.0
    if mpc_s and len(mpc_s) == len(it_times):
      per_it = max(0.0, per_it - min(mpc_s))  # Mỗi vòng chạy thử đều có thu MPC.
    n_segments = -(-LABEL_UNTIL[m] // MPC_EVERY)
    est = 1.1 * per_it * ITERATIONS + (n_segments * max(mpc_s) if mpc_s else 0.0)
    ESTIMATE_S[m] = est
    info = f"một vòng ~{per_it:.1f} s" + (f", một lần thu MPC ~{max(mpc_s):.0f} s" if mpc_s else "")
    print(f"OK {m}: chạy thử {time.time() - t0:.0f} s ({info}) → ước tính cả lần train "
          f"{est / 3600:.1f} giờ", flush=True)
  jobs_h = sum(ESTIMATE_S[m] for m in METHODS) * len(SEEDS) / 3600
  print(f"\nTổng ≈ {jobs_h:.1f} giờ GPU; với {len(GPUS)} GPU song song ≈ {jobs_h / len(GPUS):.1f} giờ. "
        f"Thời gian còn cho train: {remaining_min(TRAIN_DEADLINE) / 60:.1f} giờ.")

## 6. Train

Mỗi GPU lấy lần lượt một lần train từ hàng đợi. Lần train nào ước tính không kịp xong trước giờ
chót thì bị bỏ qua (chạy ở phiên sau); lần đang chạy mà tới giờ chót thì bị dừng. Tiến độ được in
mỗi 10 phút.

In [ ]:
import queue
import threading

jobs = queue.Queue()
for s in SEEDS:
  for m in METHODS:
    if not os.path.exists(f"{CONSOLE_DIR}/{m}_s{s}.done"):
      jobs.put((m, s))
STATUS = {}


def lane(gpu):
  while True:
    try:
      m, s = jobs.get_nowait()
    except queue.Empty:
      return
    name = f"{m}_s{s}"
    est = ESTIMATE_S.get(m)
    if est is not None and time.time() + est > TRAIN_DEADLINE:
      STATUS[name] = f"bỏ qua (cần ~{est / 3600:.1f} giờ, không đủ thời gian)"
      continue
    STATUS[name] = f"đang chạy trên GPU {gpu}"
    log = f"{CONSOLE_DIR}/{name}.log"
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": gpu, "PYTHONUNBUFFERED": "1"}
    t0 = time.time()
    with open(log, "w") as f:
      p = subprocess.Popen(train_cmd(m, s), shell=True, cwd=REPO_DIR, env=env,
                           stdout=f, stderr=subprocess.STDOUT, start_new_session=True)
      while p.poll() is None:
        if time.time() > TRAIN_DEADLINE:
          os.killpg(p.pid, 15)
          p.wait()
          break
        time.sleep(20)
    if p.returncode == 0:
      open(f"{CONSOLE_DIR}/{name}.done", "w").write(f"{time.time() - t0:.0f}\n")
      STATUS[name] = f"XONG sau {(time.time() - t0) / 3600:.2f} giờ"
    elif time.time() > TRAIN_DEADLINE:
      STATUS[name] = "dừng vì hết giờ (train lại ở phiên sau)"
    else:
      STATUS[name] = f"LỖI (mã {p.returncode}), xem {log}"


def last_progress(name):
  path = f"{CONSOLE_DIR}/{name}.log"
  if not os.path.exists(path):
    return ""
  text = ANSI.sub("", open(path, errors="ignore").read())
  its = re.findall(r"Learning iteration\s+(\d+)/", text)
  eta = re.findall(r"ETA:\s*([\d:]+)", text)
  return f"vòng {its[-1]}/{ITERATIONS}, còn {eta[-1] if eta else '?'}" if its else "đang khởi tạo"


threads = [threading.Thread(target=lane, args=(g,), daemon=True) for g in GPUS]
for t in threads:
  t.start()
last_print = 0.0
while any(t.is_alive() for t in threads):
  time.sleep(20)
  if time.time() - last_print > 600:
    last_print = time.time()
    print(time.strftime("%H:%M"), f"| còn {remaining_min(TRAIN_DEADLINE):.0f} phút cho train", flush=True)
    for name, st in sorted(STATUS.items()):
      extra = f" ({last_progress(name)})" if st.startswith("đang chạy") else ""
      print(f"  {name:>20}: {st}{extra}", flush=True)
print("\nKết thúc train:")
for name, st in sorted(STATUS.items()):
  print(f"  {name:>20}: {st}")

## 7. Đường học

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

COLORS = {"Base": "#9467bd", "BC-300": "#1f77b4", "BC-floor": "#ff7f0e"}
TAGS = {
  "Train/mean_reward": "Mean reward (lúc train)",
  "Episode_Reward/track_linear_velocity": "Bám vận tốc thẳng (tối đa 2,0)",
  "Train/mean_episode_length": "Độ dài episode (bước)",
}


def done(m, s):
  return os.path.exists(f"{CONSOLE_DIR}/{m}_s{s}.done")


def run_dir(m, s):
  found = glob.glob(f"{LOG_ROOT}/**/*_{m}_s{s}", recursive=True)
  return max(found, key=os.path.getmtime) if found else None


def load_scalars(path):
  ea = EventAccumulator(path, size_guidance={"scalars": 0})
  ea.Reload()
  have = ea.Tags()["scalars"]
  return {t: {e.step: e.value for e in ea.Scalars(t)} if t in have else {} for t in TAGS}


fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for m in METHODS:
  curves = {t: [] for t in TAGS}
  for s in SEEDS:
    d = run_dir(m, s)
    if d and done(m, s):
      sc = load_scalars(d)
      for t in TAGS:
        if sc[t]:
          curves[t].append(sc[t])
  for ax, t in zip(axes, TAGS):
    cs = curves[t]
    for c in cs:
      ks = sorted(c)
      ax.plot(ks, [c[k] for k in ks], color=COLORS[m], alpha=0.25, lw=1)
    if cs:
      common = sorted(set.intersection(*(set(c) for c in cs)))
      ax.plot(common, [np.mean([c[k] for c in cs]) for k in common], color=COLORS[m], lw=2,
              label=f"{m} ({len(cs)} seed)")
for ax, t in zip(axes, TAGS):
  ax.axvline(500, color="k", ls="--", lw=0.8)
  ax.set_xlabel("Vòng lặp")
  ax.set_ylabel(TAGS[t])
  ax.grid(alpha=0.3)
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUT_ROOT}/train_curves.png", dpi=150)
plt.show()

## 7.1. Thầy MPC có hơn trò không (Δ)

Ở mỗi lần gắn nhãn, cứ mỗi `TEACHER_GAP_EVERY` bước, planner chạy từ **cùng một trạng thái** hai
chuỗi trong 16 bước (0,32 s): kế hoạch vừa tối ưu của MPC (thầy) và policy hiện tại dùng hành
động trung bình (trò). Δ = return của thầy − return của trò.

- **Tỉ lệ thầy hơn trò** > 0,5: thầy còn dạy được; < 0,5: trò đã giỏi hơn thầy trong ngắn hạn.
- Đây là so sánh **ngắn hạn** (16 bước), không phải reward cả episode.

In [ ]:
GAP_TAGS = {
  "Loss/mpc/collect_teacher_better_frac": "Tỉ lệ trạng thái thầy hơn trò",
  "Loss/mpc/collect_teacher_gap": "Δ trung bình (return 16 bước)",
}
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
rows = []
for m in METHODS:
  for s in SEEDS:
    d = run_dir(m, s)
    if not d:
      continue
    ea = EventAccumulator(d, size_guidance={"scalars": 0})
    ea.Reload()
    have = ea.Tags()["scalars"]
    if list(GAP_TAGS)[0] not in have:
      continue
    for ax, tag in zip(axes, GAP_TAGS):
      ev = ea.Scalars(tag)
      ax.plot([e.step for e in ev], [e.value for e in ev], "o-", color=COLORS[m], ms=3,
              alpha=0.8, label=f"{m} s{s}")
    frac = ea.Scalars(list(GAP_TAGS)[0])
    below = [e.step for e in frac if e.value < 0.5]
    rows.append((f"{m} s{s}", frac[0].value, frac[-1].value, below[0] if below else None))
for ax, tag in zip(axes, GAP_TAGS):
  ax.set_xlabel("Vòng lặp")
  ax.set_ylabel(GAP_TAGS[tag])
  ax.grid(alpha=0.3)
axes[0].axhline(0.5, color="k", ls="--", lw=0.8)
axes[1].axhline(0.0, color="k", ls="--", lw=0.8)
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUT_ROOT}/teacher_gap.png", dpi=150)
plt.show()
for name, first, last, cross in rows:
  where = f"vòng {cross}" if cross is not None else "không xuống dưới 0,5 trong lúc gắn nhãn"
  print(f"{name:>22}: thầy hơn trò ở {first:.0%} trạng thái lúc đầu, {last:.0%} ở lần gắn nhãn cuối; "
        f"lần đầu < 50%: {where}")

## 8. Đo vận tốc thực tế ở 0,5 / 1,0 / 1,5 m/s

Checkpoint cuối của mỗi lần train đã xong; policy tất định điều khiển 64 robot đi thẳng ở từng
lệnh: 1 giây đầu để tăng tốc, rồi đo trong 10 giây. **Tiêu chí "bám 1,5 m/s":** vận tốc trung bình ≥ 1,2 m/s ở lệnh 1,5 m/s
và ≤ 0,25 lần ngã mỗi robot.

In [ ]:
def last_checkpoint(d):
  ckpts = glob.glob(f"{d}/model_*.pt") if d else []
  return max(ckpts, key=lambda p: int(re.search(r"model_(\d+)", p).group(1))) if ckpts else None


first = not glob.glob(f"{EVAL_DIR}/*.json")
for s in SEEDS:
  for m in METHODS:
    out = f"{EVAL_DIR}/{m}_s{s}.json"
    ckpt = last_checkpoint(run_dir(m, s))
    if os.path.exists(out) or not done(m, s) or ckpt is None:
      continue
    if remaining_min() < 10:
      print("Gần hết giờ, dừng đánh giá.")
      break
    controllers = "zero policy" if first else "policy"
    first = False
    print(f"===== {m}_s{s}: {os.path.basename(ckpt)}", flush=True)
    run(
      f"CUDA_VISIBLE_DEVICES={GPUS[0]} {UV} mpopi-eval --task {EVAL_TASK} "
      f"--device cuda:0 --num-envs {EVAL_ENVS} --steps {EVAL_STEPS} --settle-steps {EVAL_SETTLE} "
      f"--speeds {EVAL_SPEEDS} "
      f"--controllers {controllers} --checkpoint {ckpt} --out {out}",
      cwd=REPO_DIR, quiet_tables=True, check=False,
    )

In [ ]:
import json

rows = []
for path in sorted(glob.glob(f"{EVAL_DIR}/*.json")):
  name = os.path.basename(path)[:-5]
  method = name.rsplit("_s", 1)[0]
  for r in json.load(open(path))["results"]:
    rows.append({"method": method if r["controller"] == "policy" else "đứng yên", "run": name, **r})

lines = [f"{'phương pháp':>18} {'lệnh':>5} {'vận tốc':>14} {'|sai số|':>9} {'ngã/robot':>10} {'seed':>5}"]
summary = {}
for method in ["đứng yên", *METHODS]:
  for cmd in (0.5, 1.0, 1.5):
    rs = [r for r in rows if r["method"] == method and r["command"] == cmd]
    if not rs:
      continue
    v = np.array([r["speed"] for r in rs])
    spread = f" ± {v.std():.2f}" if len(v) > 1 else ""
    lines.append(f"{method:>18} {cmd:>5.1f} {v.mean():>8.2f}{spread:<6} "
                 f"{np.mean([r['abs_error'] for r in rs]):>9.2f} "
                 f"{np.mean([r['falls_per_env'] for r in rs]):>10.2f} {len(rs):>5}")
    if cmd == 1.5 and method != "đứng yên":
      summary[method] = [(r["run"], r["speed"], r["falls_per_env"]) for r in rs]
lines.append("\nTiêu chí bám 1,5 m/s (≥ 1,2 m/s và ≤ 0,25 lần ngã/robot):")
for method, runs in summary.items():
  ok = [run_name for run_name, v, f in runs if v >= 1.2 and f <= 0.25]
  lines.append(f"  {method}: {len(ok)}/{len(runs)} lần train ĐẠT")
report = "\n".join(lines)
print(report)
open(f"{OUT_ROOT}/ket_qua.txt", "w").write(report + "\n")

## 8.1. Trọng số và loss bắt chước

Trái: trọng số BC theo vòng (đúng như cấu hình của từng biến thể). Phải: loss bắt chước
(khoảng cách giữa hành động của trò và nhãn của thầy) khi trọng số còn > 0.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for m in METHODS:
  for s in SEEDS:
    d = run_dir(m, s)
    if not (d and done(m, s)):
      continue
    ea = EventAccumulator(d, size_guidance={"scalars": 0})
    ea.Reload()
    for ax, tag in zip(axes, ["Loss/mpc/bc_weight", "Loss/mpc/bc_loss"]):
      ev = [e for e in ea.Scalars(tag) if tag != "Loss/mpc/bc_loss" or e.value > 0]
      ax.plot([e.step for e in ev], [e.value for e in ev], color=COLORS[m],
              alpha=0.9 if s == SEEDS[0] else 0.5, lw=1.2, label=f"{m} s{s}")
for ax, label in zip(axes, ["Trọng số BC", "Loss BC"]):
  ax.set_xlabel("Vòng lặp")
  ax.set_ylabel(label)
  ax.grid(alpha=0.3)
axes[0].set_xlim(0, 600)  # Loss: cả lần train (BC-floor vẫn bắt chước sau vòng 600).
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUT_ROOT}/bc_weight_loss.png", dpi=150)
plt.show()

## 8.2. Bảng tổng hợp: mốc học, thời gian, vận tốc

Mốc = vòng đầu tiên reward bám vận tốc thẳng (trung bình trượt 10 vòng) đạt 1,0 / 1,3 / 1,45 / 1,5
(tối đa 2,0). Mốc 1,45 ổn định hơn 1,5 vì gần 1,5 đường học gần như nằm ngang. Phút tới mốc cộng
dồn thời gian thu dữ liệu và cập nhật. Vận tốc và sai số đo ở lệnh 1,5 m/s (mục 8).

In [ ]:
def first_hit(values, th, k=10):
  v = np.asarray(values, dtype=float)
  if len(v) < k:
    return None
  sm = np.convolve(v, np.ones(k) / k, mode="valid")
  idx = np.where(sm >= th)[0]
  return int(idx[0]) + k - 1 if len(idx) else None


THRESHOLDS = (1.0, 1.3, 1.45, 1.5)
fmt = lambda x: "--" if x is None or (isinstance(x, float) and np.isnan(x)) else f"{x:.0f}"
lines = [f"{'lần train':>14} {'tổng (phút)':>11} " + " ".join(f"{'vòng ' + str(t):>9}" for t in THRESHOLDS)
         + f" {'phút 1,45':>9} {'v 1,5':>6} {'|e| 1,5':>8} {'ngã':>5}"]
per_method = {}
for m in METHODS:
  for s in SEEDS:
    d = run_dir(m, s)
    if not (d and done(m, s)):
      continue
    ea = EventAccumulator(d, size_guidance={"scalars": 0})
    ea.Reload()
    get = lambda t: np.array([e.value for e in ea.Scalars(t)])
    track = get("Episode_Reward/track_linear_velocity")
    cum = np.cumsum(get("Perf/collection_time") + get("Perf/learning_time")) / 60
    hits = [first_hit(track, t) for t in THRESHOLDS]
    ev_path = f"{EVAL_DIR}/{m}_s{s}.json"
    ev = [r for r in json.load(open(ev_path))["results"]
          if r["controller"] == "policy" and r["command"] == 1.5] if os.path.exists(ev_path) else []
    total = float(open(f"{CONSOLE_DIR}/{m}_s{s}.done").read()) / 60
    row = {"total": total, "hits": hits, "min145": cum[hits[2]] if hits[2] is not None else np.nan,
           "v": ev[0]["speed"] if ev else np.nan, "e": ev[0]["abs_error"] if ev else np.nan,
           "f": ev[0]["falls_per_env"] if ev else np.nan}
    per_method.setdefault(m, []).append(row)
    lines.append(f"{m + '_s' + str(s):>14} {total:>11.0f} " + " ".join(f"{fmt(h):>9}" for h in hits)
                 + f" {fmt(row['min145']):>9} {row['v']:>6.2f} {row['e']:>8.3f} {row['f']:>5.2f}")
lines.append("")
lines.append("Trung bình theo biến thể:")
for m, rs in per_method.items():
  hits = [np.mean([r["hits"][i] for r in rs]) if all(r["hits"][i] is not None for r in rs) else None
          for i in range(len(THRESHOLDS))]
  lines.append(f"{m:>14} {np.mean([r['total'] for r in rs]):>11.0f} "
               + " ".join(f"{fmt(h):>9}" for h in hits)
               + f" {fmt(np.mean([r['min145'] for r in rs])):>9} {np.mean([r['v'] for r in rs]):>6.2f} "
               f"{np.mean([r['e'] for r in rs]):>8.3f} {np.mean([r['f'] for r in rs]):>5.2f}")
summary_text = "\n".join(lines)
print(summary_text)
open(f"{OUT_ROOT}/bang_tong_hop.txt", "w").write(summary_text + "\n")

## 9. Video ở lệnh 1,5 m/s (seed đầu tiên)

Nếu máy không quay được video (thiếu EGL), ô này chỉ báo lỗi, không ảnh hưởng kết quả.

In [ ]:
from IPython.display import HTML, Video, display

if RECORD_VIDEO:
  for m in METHODS:
    s = SEEDS[0]
    vdir = f"{EVAL_DIR}/video/{m}_s{s}"
    ckpt = last_checkpoint(run_dir(m, s))
    if not done(m, s) or ckpt is None or glob.glob(f"{vdir}/*.mp4") or remaining_min() < 5:
      continue
    run(
      f"CUDA_VISIBLE_DEVICES={GPUS[0]} {UV} mpopi-eval --task {EVAL_TASK} "
      f"--device cuda:0 --num-envs 1 --steps 250 --speeds 1.5 --controllers policy "
      f"--checkpoint {ckpt} --video-dir {vdir}",
      cwd=REPO_DIR, quiet_tables=True, check=False,
    )
for path in sorted(glob.glob(f"{EVAL_DIR}/video/*/*.mp4")):
  display(HTML(f"<b>{os.path.relpath(path, EVAL_DIR)}</b>"))
  display(Video(path, embed=True, width=480))

## 10. Dọn và đóng gói

Chỉ giữ checkpoint cuối của mỗi lần train (để phiên sau đánh giá lại được) và nén kết quả thành
`/kaggle/working/mpc_g1_bc_results.zip` (không kèm checkpoint). Tải về ở tab **Output**.

In [ ]:
import zipfile

for d in glob.glob(f"{LOG_ROOT}/**/", recursive=True):
  ckpts = glob.glob(f"{d}/model_*.pt")
  keep = last_checkpoint(d) if ckpts else None
  for c in ckpts:
    if c != keep:
      os.remove(c)

src = "/kaggle/working/mpc_g1_bc_results.zip"
with zipfile.ZipFile(src, "w", zipfile.ZIP_DEFLATED) as zf:
  for root, _, files in os.walk(OUT_ROOT):
    for fname in files:
      if not fname.endswith((".pt", ".onnx")):
        path = os.path.join(root, fname)
        zf.write(path, os.path.relpath(path, OUT_ROOT))
print(f"Xong. Kích thước zip: {os.path.getsize(src) / 1e6:.1f} MB")
print(f"Tổng thời gian: {(time.time() - NOTEBOOK_START) / 3600:.2f} giờ")